# NSL-KDD Dataset Audit & Exploratory Data Profiling

This notebook performs a **production-grade Exploratory Data Analysis (EDA)** and **data profiling** of the **NSL-KDD dataset** for Network Intrusion Detection Systems (NIDS). The goal of this analysis is to evaluate data quality, analyze threat taxonomies, and establish robust preprocessing guidelines before training any machine learning models.

---

## 1. Setup and Package Imports

First, we import the core data science stack: `pandas`, `numpy`, `matplotlib.pyplot`, and `seaborn`. We also import standard utility packages to support automatic downloading of datasets if they are not present locally.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set style preferences for high-quality plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

print("Libraries successfully imported.")

## 2. Robust Dataset Ingestion

To make this notebook portable and easy to run out-of-the-box, the ingestion layer automatically checks if the raw dataset exists in the standard relative location `../data/raw/KDDTrain+.txt`. If missing, it downloads it directly from the official University of New Brunswick (UNB) NSL-KDD GitHub mirror.

We also define the exact 43-column schema, including the 41 network features, the multi-class target `class` label, and the auxiliary `difficulty_score` label.

In [ ]:
# Define raw data file directories and paths
RAW_DIR = os.path.join("..", "data", "raw")
DATA_PATH = os.path.join(RAW_DIR, "KDDTrain+.txt")
DOWNLOAD_URL = "https://raw.githubusercontent.com/defcom17/NSL-KDD/master/KDDTrain+.txt"

# Ensure local directories exist
os.makedirs(RAW_DIR, exist_ok=True)

# Download the dataset if not present locally
if not os.path.exists(DATA_PATH):
    print(f"Dataset not found at {DATA_PATH}. Downloading from official mirror...")
    try:
        urllib.request.urlretrieve(DOWNLOAD_URL, DATA_PATH)
        print("Download complete!")
    except Exception as e:
        print(f"Failed to download dataset: {e}")
else:
    print(f"Dataset loaded locally from: {DATA_PATH}")

# NSL-KDD 43 columns schema
column_names = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_hot_login", "is_guest_login",
    "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "class", "difficulty_score"
]

# Load the dataset into memory
df = pd.read_csv(DATA_PATH, names=column_names, header=None)
print(f"\nSuccessfully loaded {df.shape[0]} records with {df.shape[1]} columns.")

## 3. Basic Structural Audit (Shape and Types)

Here, we inspect the loaded DataFrame structure. We will check the dataset dimensions (`shape`), output a summary of data types (`dtypes`), and inspect a small sample of records to verify correct parsing.

In [ ]:
print("=== DATASET SHAPE ===")
print(f"Total Connections: {df.shape[0]}")
print(f"Total Schema Attributes: {df.shape[1]}\n")

print("=== COLUMN DATA TYPES AND COUNTS ===")
print(df.dtypes.value_counts())

print("\n=== OBJECT / CATEGORICAL COLUMNS ===")
print(df.select_dtypes(include=['object']).columns.tolist())

print("\n=== SAMPLE DATA PREVIEW ===")
df.head(3)

## 4. Integrity Checks: Missing Values & Duplicate Records

An essential stage of the audit is checking data integrity. In real production setups, missing values or high sample duplicate ratios can degrade training efficiency and lead to overly optimistic (biased) evaluations. Here, we calculate these counts explicitly.

In [ ]:
# Calculate missing values per column
missing_values = df.isnull().sum()
total_missing = missing_values.sum()
print("=== INTEGRITY AUDIT: MISSING VALUES ===")
print(f"Total Missing Fields detected: {total_missing}")
if total_missing > 0:
    print(missing_values[missing_values > 0])
else:
    print("Excellent: No null or missing values found in the dataset.\n")

# Calculate duplicate records
# Note: We exclude the 'difficulty_score' from duplication checks as it is post-hoc.
duplicate_count = df.duplicated(subset=df.columns[:-1]).sum()
print("=== INTEGRITY AUDIT: DUPLICATE CONNECTIONS ===")
print(f"Total Duplicate Records (excluding difficulty score): {duplicate_count}")
print(f"Percentage of duplicates: {duplicate_count / len(df) * 100:.2f}%")

## 5. Descriptive Statistical Profiling

We generate descriptive statistics for numerical columns to understand their distributions, scale, and dispersion, followed by a separate analysis of our categorical predictors.

In [ ]:
print("=== NUMERICAL DESCRIPTIVE STATISTICS (SAMPLE FEATURES) ===")
# Showing statistics for key volumetric traffic columns to inspect boundaries and outliers
sample_numeric_cols = ["duration", "src_bytes", "dst_bytes", "count", "serror_rate", "dst_host_srv_count"]
display(df[sample_numeric_cols].describe())

print("\n=== CATEGORICAL VARIABLE UNIQUE VALUE COUNTS ===")
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    print(f"- Column '{col}': {df[col].nunique()} unique categories")

## 6. Threat Target Taxonomy & Imbalance Analysis

The NSL-KDD dataset's target variable `class` has two main perspectives:
1. **Binary Classification**: Normal vs. Anomaly.
2. **Multi-class Classification**: Mapping specific attack signatures to one of the **4 core attack categories** (Denial of Service - DoS, Probing, Remote-to-Local - R2L, User-to-Root - U2R).

Here, we map the exact attack types to their corresponding categories and analyze their distributions.

In [ ]:
# Mapping dictionary from NSL-KDD attack subtypes to the 4 core categories
attack_mapping = {
    # DoS attacks
    'neptune': 'DoS', 'back': 'DoS', 'smurf': 'DoS', 'teardrop': 'DoS', 'pod': 'DoS', 'land': 'DoS',
    'apache2': 'DoS', 'mailbomb': 'DoS', 'processtable': 'DoS', 'udpstorm': 'DoS',
    # Probing / Reconnaissance
    'satan': 'Probe', 'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'mscan': 'Probe', 'saint': 'Probe',
    # Remote-to-Local (R2L)
    'guess_passwd': 'R2L', 'ftp_write': 'R2L', 'imap': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'multihop': 'R2L', 'phf': 'R2L', 'spy': 'R2L', 'sendmail': 'R2L', 'named': 'R2L', 'snmpgetattack': 'R2L',
    'snmpguess': 'R2L', 'xlock': 'R2L', 'xsnoop': 'R2L', 'worm': 'R2L',
    # User-to-Root (U2R)
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'rootkit': 'U2R', 'perl': 'U2R', 'sqlattack': 'U2R',
    'ps': 'U2R', 'xterm': 'U2R',
    # Legitimate traffic
    'normal': 'Normal'
}

# Create new taxonomic mapping column
df['attack_category'] = df['class'].map(attack_mapping)

print("=== TARGET DISTRIBUTION: BINARY LABELS ===")
binary_dist = df['class'].apply(lambda x: 'Normal' if x == 'normal' else 'Anomaly').value_counts()
print(binary_dist)
print(df['class'].apply(lambda x: 'Normal' if x == 'normal' else 'Anomaly').value_counts(normalize=True) * 100)

print("\n=== TARGET DISTRIBUTION: 5-CLASS CATEGORIES ===")
class_dist = df['attack_category'].value_counts()
class_dist_pct = df['attack_category'].value_counts(normalize=True) * 100
class_summary = pd.DataFrame({'Count': class_dist, 'Percentage (%)': class_dist_pct})
display(class_summary)

In [ ]:
# Plot Target Label Distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Binary label pie chart
axes[0].pie(binary_dist, labels=binary_dist.index, autopct='%1.1f%%', colors=['#4d94ff', '#ff4d4d'], startangle=140, explode=(0.05, 0))
axes[0].set_title("Binary Label Distribution (Normal vs. Anomaly)", fontweight='bold')

# Attack Family bar chart
sns.barplot(x=class_dist.index, y=class_dist.values, ax=axes[1], palette="viridis")
for i, val in enumerate(class_dist.values):
    axes[1].text(i, val + 500, f"{val}\n({class_dist_pct.values[i]:.2f}%)", ha='center', fontsize=10, fontweight='bold')
axes[1].set_yscale('log') # Log scale to handle massive class imbalance visibility
axes[1].set_title("Network Threat Categories (Log Scaled)", fontweight='bold')
axes[1].set_xlabel("Threat Group")
axes[1].set_ylabel("Number of Connections (Log Scale)")

plt.tight_layout()
plt.show()

## 7. Correlation Analysis & Feature Redundancy

To detect feature redundancy (multicollinearity), we run a Pearson correlation analysis across numerical columns. We focus on the relationship among time-based and host-based error rates, which frequently track the same network anomalies.

In [ ]:
# Extract subset of continuous attributes representing traffic statistics and rates
traffic_rate_cols = [
    "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate"
]

corr_matrix = df[traffic_rate_cols].corr()

# Set up a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, cbar_kws={"shrink": .8})
plt.title("Multi-Collinearity Heatmap: Network Traffic & Rate Features", fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Univariate Distributions and Feature Skewness

Network payload measurements like `src_bytes` (data sent from client) often span massive ranges, causing highly skewed distributions. Here, we analyze these distributions and demonstrate how a logarithmic transformation helps resolve this skewness.

In [ ]:
# Calculate raw skewness for key features
print(f"Raw Skewness of 'src_bytes': {df['src_bytes'].skew():.2f}")
print(f"Raw Skewness of 'dst_bytes': {df['dst_bytes'].skew():.2f}")
print(f"Raw Skewness of 'duration':  {df['duration'].skew():.2f}")

# Compare original vs. log-transformed distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Original Skewed Plot
sns.histplot(df['src_bytes'], bins=50, ax=axes[0], color='#ff6666', kde=False)
axes[0].set_yscale('log')
axes[0].set_title("Original 'src_bytes' (Log-y Scale due to skewness)", fontweight='bold')
axes[0].set_xlabel("Raw Source Bytes")

# Log Transformed Plot
log_src_bytes = np.log1p(df['src_bytes'])
sns.histplot(log_src_bytes, bins=50, ax=axes[1], color='#66cc99', kde=True)
axes[1].set_title("Log-Transformed Log1p('src_bytes')", fontweight='bold')
axes[1].set_xlabel("Log(Bytes + 1)")

plt.tight_layout()
plt.show()

## 9. Categorical Variable Deep Dive

Now we analyze the categorical predictors. We plot the distribution of protocols (`protocol_type`) and network connection flags (`flag`) to understand their patterns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Protocol Type Distribution
sns.countplot(data=df, x='protocol_type', hue='attack_category', ax=axes[0], palette="Set2")
axes[0].set_yscale('log')
axes[0].set_title("Protocol Types Across Threat Families (Log Scaled)", fontweight='bold')
axes[0].set_xlabel("Protocol Type")
axes[0].set_ylabel("Count (Log Scale)")

# Flag Distribution
sns.countplot(data=df, x='flag', order=df['flag'].value_counts().index, ax=axes[1], palette="muted")
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_title("Frequency of Connection Flags", fontweight='bold')
axes[1].set_xlabel("Connection Flag State")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 10. Data Quality Audit Findings Summary

Based on this data profiling analysis, we have identified several key details to guide our model development pipeline:

1. **Drop Constant Columns**: The column `num_outbound_cmds` has exactly **zero variance** (all values are $0.0$). We should explicitly drop it from our preprocessing steps.
2. **Handle Extreme Skewness**: Byte volume columns (`src_bytes`, `dst_bytes`) require a **logarithmic transformation** (`np.log1p`) to help linear algorithms and neural networks converge.
3. **Use Robust Scaling**: Continuous counter columns (`count`, `srv_count`, etc.) contain extreme outliers due to flood events. We should use **RobustScaler** (median and IQR-based) rather than StandardScaler to prevent compression issues.
4. **Mitigate Class Imbalance**: The dataset exhibits severe class imbalance, particularly for `U2R` (0.04%) and `R2L` (0.75%). Standard classification metrics like accuracy will be misleading; we must use **Precision, Recall, and F1-Scores per class**, and consider class weights during training.
5. **One-Hot Encoding Expansion**: One-hot encoding `protocol_type` and `flag` is appropriate, but the `service` column has over 70 unique values. We should group rare service types into an `'other'` category to limit feature expansion.
6. **Prevent Data Leakage**: The auxiliary `difficulty_score` column must be dropped before training as it is a post-hoc evaluation score that would not be available in a live network deployment.